In [1]:
!pip install pyspark

In [2]:
import os
import shutil
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, lower, initcap, when,
    count, isnan, to_date, regexp_replace
)
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType
)

spark = SparkSession.builder \
    .appName("Sales Data Lake Preprocessing") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
# Create the data lake folders
os.makedirs("data_lake/bronze", exist_ok=True)
os.makedirs("data_lake/silver/preprocessed_sales", exist_ok=True)

# Copy uploaded raw file into Bronze
source_file = "/content/sales.csv"
bronze_file = "data_lake/bronze/sales.csv"

shutil.copy2(source_file, bronze_file)

print("Data Lake structure created!")

Data Lake structure created!


In [4]:
# Read the raw dataset using Spark
df = spark.read.csv(
    bronze_file,
    header=True,
    inferSchema=True
)

print("Dataset Schema:")
df.printSchema()

print("First 5 rows:")
df.show(5, truncate=False)

print("Total records:", df.count())

print("Column names:", df.columns)

Dataset Schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

First 5 rows:
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+--------+--------+----------+------------

In [5]:
from pyspark.sql.functions import lit
from pyspark.sql.functions import (
    col, count, isnan, trim, lower, length,
    regexp_replace, to_date, coalesce
)

# 1. Identify duplicate rows
duplicate_count = df.count() - df.dropDuplicates().count()
print("Duplicate rows:", duplicate_count)

# 2. Identify null and empty values in every column
null_counts = df.select([
    count(
        when(
            col(c).isNull() |
            (trim(col(c).cast("string")) == "") |
            (isnan(col(c)) if str(df.schema[c].dataType) in ["DoubleType", "FloatType"] else lit(False)),
            c
        )
    ).alias(c)
    for c in df.columns
])

print("Null or empty values:")
null_counts.show(truncate=False)

# 3. Inspect text-format inconsistencies
for field in df.columns:
    if isinstance(df.schema[field].dataType, StringType):
        print(f"Inconsistent text values in {field}:")
        df.select(field).distinct().show(10, truncate=False)

Duplicate rows: 10
Null or empty values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|0       |0          |10           |0      |0       |0       |0         |0               |8             |8   |0    |0         |0           |
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Inconsistent text values in customer_id:
+-----------+
|customer_id|
+-----------+
|C0114      |
|C0141      |
|C0243      |
|C0034      |
|C0154      |
|C0347      |
|C0130      |
|C0338      |
|C0167      |
|C0179      |
+-----------+
only showing top

In [6]:
# Identify columns based on their names
quantity_cols = [c for c in df.columns if "quantity" in c.lower() or c.lower() == "qty"]
price_cols = [c for c in df.columns if "price" in c.lower()]
discount_cols = [c for c in df.columns if "discount" in c.lower()]
date_cols = [c for c in df.columns if "date" in c.lower()]

print("Quantity columns:", quantity_cols)
print("Price columns:", price_cols)
print("Discount columns:", discount_cols)
print("Date columns:", date_cols)

# Inspect numerical columns
for c in quantity_cols + price_cols + discount_cols:
    print(f"Checking {c}")
    df.select(c).describe().show()

# Inspect date values
for c in date_cols:
    print(f"Checking date formats in {c}")
    df.select(c).distinct().show(20, truncate=False)

Quantity columns: ['quantity']
Price columns: ['unit_price']
Discount columns: ['discount_percent']
Date columns: ['order_date']
Checking quantity
+-------+------------------+
|summary|          quantity|
+-------+------------------+
|  count|              1000|
|   mean|             5.355|
| stddev|2.9916425128890194|
|    min|                -2|
|    max|                10|
+-------+------------------+

Checking unit_price
+-------+------------------+
|summary|        unit_price|
+-------+------------------+
|  count|              1000|
|   mean|            5702.4|
| stddev|10990.194484849431|
|    min|              -500|
|    max|             81010|
+-------+------------------+

Checking discount_percent
+-------+------------------+
|summary|  discount_percent|
+-------+------------------+
|  count|              1000|
|   mean|            19.948|
| stddev|14.236782711763993|
|    min|                -5|
|    max|               120|
+-------+------------------+

Checking date formats

In [7]:

df_clean = df.dropDuplicates()

print("Rows before removing duplicates:", df.count())
print("Rows after removing duplicates:", df_clean.count())

Rows before removing duplicates: 1000
Rows after removing duplicates: 990


In [8]:
from pyspark.sql.functions import initcap

df_clean = df_clean.select([
    (
        initcap(trim(col(c)))
        if isinstance(df_clean.schema[c].dataType, StringType)
        and any(word in c.lower() for word in [
            "name", "city", "category", "product", "region", "country"
        ])
        else trim(col(c))
        if isinstance(df_clean.schema[c].dataType, StringType)
        else col(c)
    ).alias(c)
    for c in df_clean.columns
])

df_clean.show(5, truncate=False)

+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6       |610       |12              |UPI      

In [9]:
from pyspark.sql.functions import when, col

# Apply validation rules
for c in quantity_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c).cast("double") > 0, col(c).cast("double"))
        .otherwise(None)
    )

for c in price_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c).cast("double") >= 0, col(c).cast("double"))
        .otherwise(None)
    )

for c in discount_cols:
    df_clean = df_clean.withColumn(
        c,
        when(
            (col(c).cast("double") >= 0) &
            (col(c).cast("double") <= 100),
            col(c).cast("double")
        ).otherwise(None)
    )

# Impute missing numerical values using the median
numeric_cols = quantity_cols + price_cols + discount_cols

for c in numeric_cols:
    median_values = df_clean.approxQuantile(c, [0.5], 0.01)

    if median_values:
        df_clean = df_clean.fillna({c: median_values[0]})

# Fill missing descriptive text values
for c in df_clean.columns:
    if isinstance(df_clean.schema[c].dataType, StringType):
        df_clean = df_clean.fillna({c: "Unknown"})

df_clean.show(5, truncate=False)

+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6.0     |610.0     |12.0            |UPI      

In [10]:
from pyspark.sql.functions import (
    col, coalesce, try_to_timestamp, date_format, lit
)

# Safely parse dates without failing on invalid values
spark.conf.set("spark.sql.ansi.enabled", "false")

date_formats = [
    "yyyy-MM-dd",
    "dd-MM-yyyy",
    "MM/dd/yyyy",
    "dd/MM/yyyy",
    "yyyy/MM/dd",
    "dd-MMM-yyyy"
]

for c in date_cols:
    date_string = col(c).cast("string")

    parsed_date = coalesce(*[
        try_to_timestamp(date_string, lit(fmt)).cast("date")
        for fmt in date_formats
    ])

    df_clean = df_clean.withColumn(
        c,
        date_format(parsed_date, "yyyy-MM-dd")
    )

print("Date preprocessing completed!")
df_clean.show(5, truncate=False)

Date preprocessing completed!
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6.0     |610.0  

In [11]:
from pyspark.sql.functions import col, count, when

print("Original row count:", df.count())
print("Preprocessed row count:", df_clean.count())

print("Final schema:")
df_clean.printSchema()

print("Remaining missing values:")

df_clean.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df_clean.columns
]).show(truncate=False)

Original row count: 1000
Preprocessed row count: 990
Final schema:
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = false)
 |-- customer_name: string (nullable = false)
 |-- product: string (nullable = false)
 |-- category: string (nullable = false)
 |-- quantity: double (nullable = false)
 |-- unit_price: double (nullable = false)
 |-- discount_percent: double (nullable = false)
 |-- payment_method: string (nullable = false)
 |-- city: string (nullable = false)
 |-- state: string (nullable = false)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = false)

Remaining missing values:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+---

In [12]:
# Save processed data in the Silver layer

silver_folder = "data_lake/silver/preprocessed_sales"

df_clean.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(silver_folder)

print("Silver data saved successfully!")

Silver data saved successfully!


In [13]:
import glob
import shutil
import os

# Find the Spark-generated CSV file
part_files = glob.glob(
    "data_lake/silver/preprocessed_sales/part-*.csv"
)

# Create submission folder
os.makedirs("submission/silver", exist_ok=True)

# Copy the Spark output to a single CSV file
if part_files:
    shutil.copy(
        part_files[0],
        "submission/silver/sales_silver.csv"
    )
    print("sales_silver.csv created successfully!")
else:
    print("No CSV part file found. Run Step 12 again.")

sales_silver.csv created successfully!


In [14]:
# Read the saved CSV using Spark
silver_df = spark.read.csv(
    "submission/silver/sales_silver.csv",
    header=True,
    inferSchema=True
)

print("Silver dataset:")
silver_df.show(5, truncate=False)

print("Silver row count:", silver_df.count())

print("Silver schema:")
silver_df.printSchema()

Silver dataset:
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports   |2.0     |670.0     |30.0            |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books    |4.0     |670.0     |0.0             |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books    |6.0     |610.0     |12.0      

In [15]:
from google.colab import files

files.download("submission/silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>